# Classification evaluation

Evaluate fixed JSONL artifacts on CPU. Development rows support tuning; test rows describe held-out RELLIS recordings on one campus. Public results remain provisional for small-hazard coverage; current availability of the six planned phone clips is reported below. Regions are not independent scenes; material-policy agreement does not establish physical traversability.

Change the configuration below before running. No weights or datasets are downloaded; exports are opt-in.

In [ ]:
import json
import os
from pathlib import Path
import sys
from copy import deepcopy
from html import escape

working_dir = Path.cwd().resolve()
candidates = [path for base in (working_dir, *working_dir.parents) for path in (base, base / 'VLM_evaluation')]
REPOSITORY = next((p for p in candidates if (p / 'docs/interfaces.md').is_file() and (p / 'src').is_dir()), None)
if REPOSITORY is None:
    raise RuntimeError('Start the notebook inside VLM_evaluation or its repository checkout.')
sys.path.insert(0, str(REPOSITORY / 'src'))

# Inputs only: keep generated results under the destination run/evaluation.
DATA_ROOT = Path(os.environ.get('TRAVERSABILITY_DATA_ROOT', str(REPOSITORY / 'data')))
RUN_ROOT = Path(os.environ.get('TRAVERSABILITY_RUN_ROOT', str(REPOSITORY / 'runs')))
RUN_NAME = 'public_rellis_material_v1'
MODEL_KEYS = ['clip_vit_b32', 'qwen3_vl_4b', 'qwen3_5_4b']
EXPECTED_METADATA = {}  # JSON-pointer/value checks, e.g. {'/configuration/settings/seed': 0}
EXPECTED_METADATA_SHA256 = {}  # Optional canonical metadata hashes keyed by model_key.
COMPARISONS = []  # Optional [{run_name, model_key, configuration_label, expected_metadata}].
REVIEW_LABELS_PATH = None  # Optional audit JSONL; export never overwrites it.
FIXTURE = False
EXPORT = False
PLOTS = False  # Requires optional matplotlib.

config = deepcopy(json.loads((REPOSITORY / 'configs/evaluation/default.json').read_text(encoding='utf-8')))
config.update(data_root=str(DATA_ROOT), run_root=str(RUN_ROOT), run_name=RUN_NAME,
              model_keys=MODEL_KEYS, fixture=FIXTURE)
config['comparisons'] = deepcopy(COMPARISONS) if COMPARISONS else [
    dict(run_name=RUN_NAME, model_key=key, configuration_label=key,
         expected_metadata=deepcopy(EXPECTED_METADATA),
         **({'expected_metadata_sha256': EXPECTED_METADATA_SHA256[key]} if key in EXPECTED_METADATA_SHA256 else {}))
    for key in MODEL_KEYS
]
if REVIEW_LABELS_PATH is not None:
    config['review_labels_path'] = str(REVIEW_LABELS_PATH)

try:
    from PIL import Image
    from IPython.display import HTML, display
    from traversability_evaluation import evaluate, export_report
    from traversability_evaluation.reporting import comparison_html, render_review_gallery
except ModuleNotFoundError as exc:
    raise RuntimeError('Install requirements/evaluation.txt and use a notebook kernel with IPython; no inference environment is required.') from exc

print(json.dumps(config, indent=2))

## Validate identity and coverage

Missing or duplicate IDs, split/model/configuration mismatch and unusable masks are explicit. Pending references are excluded and counted; completed reference-unknown labels remain eligible. A missing primary prediction blocks all splits of its configuration.

In [ ]:
report = evaluate(config)
display(HTML('<h3>Validation</h3><pre>' + escape(json.dumps(report['validation'], indent=2)) + '</pre>'))
display(HTML('<h3>Completeness: expected versus available</h3><pre>' + escape(json.dumps(report['coverage'], indent=2)) + '</pre>'))
if report.get('fixture'):
    display(HTML('<strong>SYNTHETIC FIXTURE: do not use as experiment coverage.</strong>'))
phone = report.get('coverage', {}).get('phone_hazard_clips', {})
phone_notice = ('Phone hazard clips: ' + str(phone.get('available_recordings', 0)) + '/' + str(phone.get('expected', 6)) + ' available; ' + str(phone.get('not_available', 6)) + ' pending; status=' + str(phone.get('status', 'unavailable')) + '. Small-hazard coverage=' + str(phone.get('small_hazard_coverage', 'not_established')) + '.')
display(HTML('<p>' + escape(phone_notice) + '</p>'))

## Configuration comparison and per-source results

Each rate shows numerator/denominator. Error-status predictions remain unknown and count as failures. Empty denominators give null; unavailable metrics must not be interpreted as zero.

In [ ]:
display(HTML(comparison_html(report)))
if report.get('per_source'):
    display(HTML('<h3>Per-source breakdown</h3>' + comparison_html(dict(report, comparison=report['per_source']))))

## Three-way confusion matrices

Reference labels form rows; predictions form columns. Display all-source and per-source tables separately for development and held-out test.

In [ ]:
labels = ['traversable', 'non_traversable', 'unknown']
seen = set()
for row in report.get('comparison', []) + report.get('per_source', []):
    identity = tuple(row.get(key) for key in ('run_name', 'model_key', 'configuration_label', 'split', 'source'))
    if identity in seen:
        continue
    seen.add(identity)
    title = ' / '.join(str(value) for value in identity)
    matrix = row.get('confusion_matrix')
    if matrix is None:
        display(HTML('<p>' + escape(title) + ': confusion unavailable (' + escape(row.get('status', '')) + ').</p>'))
        continue
    header = '<tr><th>Reference / prediction</th>' + ''.join('<th>' + label + '</th>' for label in labels) + '</tr>'
    body = ''.join('<tr><th>' + labels[i] + '</th>' + ''.join('<td>' + str(count) + '</td>' for count in counts) + '</tr>' for i, counts in enumerate(matrix))
    display(HTML('<h4>' + escape(title) + '</h4><table border="1" cellpadding="5">' + header + body + '</table>'))

## Separate mask/hazard diagnostic and small review gallery

Annotated-hazard-overlap acceptance includes mixed masks and remains separate from primary classification. Review suggestions are tentative. Optional human audit is capped at 20 cases; unsupported causal assignments remain unreviewed.

In [ ]:
display(HTML('<h3>Mask exclusions</h3><pre>' + escape(json.dumps(report.get('mask_breakdown', []), indent=2)) + '</pre>'))
display(HTML('<h3>Annotated-hazard-overlap acceptance (separate diagnostic)</h3><pre>' + escape(json.dumps(report.get('hazard_overlap', []), indent=2)) + '</pre>'))
display(HTML(render_review_gallery(report, limit=20)))

## Optional export

Set `EXPORT = True` in the top cell to save compact report/coverage JSON, comparison/per-source/confusion/error CSVs and the review HTML beneath the destination run's `evaluation/`. Optional plots require `PLOTS = True`. No generated result belongs in Git.

In [ ]:
if EXPORT:
    exported_paths = export_report(report, plots=PLOTS)
    print(json.dumps(exported_paths, indent=2))
else:
    print('Export disabled; set EXPORT=True in the configuration cell to save artifacts.')